# Phase 2 — XGBoost Model: Hypertension Prediction (RIPAS Dataset)
**Triage IQ — Predicting Hypertension from Emergency Department Triage Data**

This notebook develops and evaluates an optimised XGBoost model on the RIPAS ED triage dataset.  
It constitutes the primary Phase 2 contribution of the project.

**Dataset:** `ripas_dataset.csv`  
**Target variable:** `HTN/CHD` (clinically recorded, binary)  
**Key challenges:** ~96% class imbalance, single-centre dataset (n = 1,636)

Pipeline:
1. Data cleaning and preprocessing
2. SMOTE applied inside training folds only
3. Repeated stratified CV for stable baseline estimates
4. Hyperparameter tuning: RandomizedSearchCV → GridSearchCV
5. Threshold optimisation (0.35) to maximise recall
6. SHAP interpretability

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RandomizedSearchCV, GridSearchCV,
    RepeatedStratifiedKFold, cross_validate
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("ripas_dataset.csv")

print("Shape:", df.shape)
df.head()

## 3. Data Cleaning

- `age`: stored as strings with trailing `'Y'` — stripped and cast to numeric
- Vital-sign columns: cast to numeric; invalid strings become `NaN` for imputation
- Binary comorbidity / outcome flags: mapped from string labels to 0/1
- Gender: encoded as `M=1`, `F=0`

In [ ]:
# Strip trailing 'Y' from age strings (e.g. '45Y' -> 45)
df['age'] = pd.to_numeric(
    df['age'].astype(str).str.replace('Y', '', regex=False),
    errors='coerce'
)

# Convert vital-sign and stay columns to numeric
numeric_cols = [
    'Blood pressure diastole',
    'blood pressure/systolic',
    'pulse rate',
    'respiratory rate',
    'SPO2',
    'Temperature',
    'PAIN SCORE',
    'STAYS IN DAYS'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Encode binary comorbidity / outcome flags (NO -> 0, named value -> 1)
binary_cols = ['HTN/CHD', 'DM', 'CKD', 'ADMISSION', 'ICU', 'NIV/VENT', 'INOTROPE', 'DEATH']
for col in binary_cols:
    df[col] = df[col].replace({'NO': 0, 'HTN/CHD': 1, 'DM': 1, 'CKD': 1})
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Encode gender
df['gender'] = df['gender'].map({'M': 1, 'F': 0})

print("Cleaned dataset shape:", df.shape)
df.head()

## 4. Define Target Variable

Rows where the `HTN/CHD` label is missing are dropped before splitting.  
The class distribution confirms the severe imbalance (~96% non-hypertensive).

In [ ]:
# Drop rows where the target label is missing
df = df[df['HTN/CHD'].notna()].reset_index(drop=True)

print("Remaining NaN in target:", df['HTN/CHD'].isna().sum())

X = df.drop('HTN/CHD', axis=1)
y = df['HTN/CHD']

print("\nClass distribution:")
print(y.value_counts())
print("\nClass proportions:")
print(y.value_counts(normalize=True).round(3))

## 5. Train/Test Split

An 80/20 stratified split is used (vs 70/30 in Phase 1) because the RIPAS dataset is small —  
keeping more training data is important for SMOTE to generate meaningful synthetic samples.  
The test set is never touched by SMOTE or any preprocessing fit.

In [ ]:
# 80/20 stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print("\nTraining class distribution:")
print(y_train.value_counts())

## 6. Class Distribution and SMOTE Effect

SMOTE is applied **inside** the ImbPipeline during training only.  
The visualisation below illustrates the intended resampling — it is for inspection only and does not affect the training pipeline.

In [ ]:
# Visualise class imbalance before any resampling
fig, ax = plt.subplots(figsize=(5, 3))
y_train.value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xticklabels(['No HTN', 'HTN'], rotation=0)
ax.set_title("Training Set Class Distribution (Before SMOTE)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = ['site of pain']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler())
        ]), numerical_cols),

        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

In [ ]:
# Illustrate SMOTE effect by fitting preprocessor + SMOTE on training data
X_pre = preprocessor.fit_transform(X_train)
_, y_res = SMOTE(sampling_strategy=0.8, random_state=42).fit_resample(X_pre, y_train)

print("Class distribution after SMOTE:")
print(pd.Series(y_res).value_counts())

fig, ax = plt.subplots(figsize=(5, 3))
pd.Series(y_res).value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xticklabels(['No HTN', 'HTN'], rotation=0)
ax.set_title("Training Set Class Distribution (After SMOTE, sampling_strategy=0.8)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 7. XGBoost Pipeline Definition

SMOTE `sampling_strategy=0.8` generates synthetic minority samples up to 80% of the majority class count — a deliberate choice to avoid over-correcting on a dataset this small.

In [ ]:
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(sampling_strategy=0.8, random_state=42)),
    ('model',        xgb)
])

## 8. Baseline Evaluation

The pipeline is evaluated with default XGBoost parameters to establish a performance floor.  
`RepeatedStratifiedKFold` (5 splits × 5 repeats = 25 evaluations) gives stable estimates for this small dataset.

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scoring = {'recall': 'recall', 'roc_auc': 'roc_auc', 'precision': 'precision', 'f1': 'f1'}

cv_results = cross_validate(pipeline, X_train, y_train, cv=rskf, scoring=scoring)

print("Repeated Stratified CV — Baseline XGBoost:")


In [ ]:
for key in cv_results:
    if key.startswith("test_"):
        mean_score = np.mean(cv_results[key])
        std_score  = np.std(cv_results[key])
        print(f"{key[5:]:12s}: {mean_score:.4f} ± {std_score:.4f}")

In [ ]:
# Fit baseline on full training set and evaluate on test set
pipeline.fit(X_train, y_train)
y_prob_base  = pipeline.predict_proba(X_test)[:, 1]
y_pred_base_05 = pipeline.predict(X_test)                         # default threshold
y_pred_base_35 = (y_prob_base >= 0.35).astype(int)               # clinical threshold

for label, y_pred in [("Threshold = 0.5", y_pred_base_05), ("Threshold = 0.35", y_pred_base_35)]:
    print(f"--- BASELINE {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_base):.3f}")
    print(classification_report(y_test, y_pred))

## 9. Hyperparameter Tuning

### 9a. RandomizedSearchCV (broad exploration)

10-fold stratified CV with `scoring='recall'` — recall is the primary clinical metric  
because missing a hypertensive patient at triage carries greater risk than a false positive.

In [ ]:
cv_rand = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

param_dist = {
    'model__n_estimators':  [100, 200, 300],
    'model__max_depth':     [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__subsample':     [0.8, 1.0]
}

rand_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv_rand,
    scoring='recall',
    n_jobs=-1,
    random_state=42,
    verbose=0
)
rand_search.fit(X_train, y_train)

print("Best recall (RandomCV):", round(rand_search.best_score_, 3))
print("Best parameters:",        rand_search.best_params_)

In [ ]:
# Evaluate RandomCV best model
rand_best  = rand_search.best_estimator_
y_prob_rand = rand_best.predict_proba(X_test)[:, 1]
y_pred_rand_05 = rand_best.predict(X_test)
y_pred_rand_35 = (y_prob_rand >= 0.35).astype(int)

for label, y_pred in [("Threshold = 0.5", y_pred_rand_05), ("Threshold = 0.35", y_pred_rand_35)]:
    print(f"--- RandomCV {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_rand):.3f}")
    print(classification_report(y_test, y_pred))

### 9b. GridSearchCV (fine-tuned around best RandomCV region)

In [ ]:
cv_grid = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'model__n_estimators':     [150, 200, 250],
    'model__max_depth':        [3, 4, 5],
    'model__learning_rate':    [0.05, 0.1],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv_grid,
    scoring='recall',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print("Best recall (GridSearch):", round(grid_search.best_score_, 3))
print("Best parameters:",          grid_search.best_params_)

### 9c. GridSearchCV — Test Set Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_prob     = best_model.predict_proba(X_test)[:, 1]

y_pred_default  = best_model.predict(X_test)
y_pred_adjusted = (y_prob >= 0.35).astype(int)

for label, y_pred in [("Threshold = 0.5", y_pred_default), ("Threshold = 0.35", y_pred_adjusted)]:
    print(f"--- GridSearch {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")
    print(classification_report(y_test, y_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_pred, title in zip(
    axes,
    [y_pred_default, y_pred_adjusted],
    ["XGBoost GridSearch (Threshold 0.5)", "XGBoost GridSearch (Threshold 0.35)"]
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=["No HTN", "HTN"]
    ).plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — XGBoost (GridSearch)")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Threshold Analysis

The precision-recall trade-off is evaluated across a range of thresholds to justify the 0.35 selection.  
In a screening context, the cost of a missed hypertensive patient (false negative) outweighs the cost of a false positive.

In [ ]:
thresholds = [0.5, 0.45, 0.40, 0.35, 0.30, 0.25]

threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    threshold_results.append({
        "Threshold": t,
        "Recall":    round(report.get("1.0", report.get(1, {})).get("recall",    0), 3),
        "Precision": round(report.get("1.0", report.get(1, {})).get("precision", 0), 3),
        "F1":        round(report.get("1.0", report.get(1, {})).get("f1-score",  0), 3),
        "Accuracy":  round(accuracy_score(y_test, y_pred_t), 3)
    })

threshold_df = pd.DataFrame(threshold_results)
print("Threshold sweep results:")
display(threshold_df)

In [ ]:
# Visualise recall vs precision across thresholds
plt.figure(figsize=(7, 4))
plt.plot(threshold_df["Threshold"], threshold_df["Recall"],    marker='o', label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker='s', label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["F1"],        marker='^', label="F1")
plt.axvline(0.35, color='red', linestyle='--', alpha=0.7, label="Selected threshold (0.35)")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Recall / Precision / F1 vs Classification Threshold")
plt.legend()
plt.tight_layout()
plt.show()

## 11. SHAP Interpretability

SHAP (SHapley Additive Explanations) is applied to the best GridSearch model to identify which triage features most influence predictions.  
This is essential for clinical adoption — a clinician needs to understand *why* a patient is flagged before acting on the output.

In [ ]:
# Transform training features through the preprocessor only
X_transformed = best_model.named_steps['preprocessor'].transform(X_train)
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

# Reconstruct feature names after one-hot encoding
cat_features_out = (
    best_model.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(categorical_cols)
)
feature_names = list(numerical_cols) + list(cat_features_out)
print(f"Total features after encoding: {len(feature_names)}")

In [ ]:
explainer       = shap.TreeExplainer(best_model.named_steps['model'])
raw_shap_values = explainer.shap_values(X_transformed)

shap_explanation = shap.Explanation(
    values=raw_shap_values,
    base_values=explainer.expected_value,
    data=X_transformed,
    feature_names=feature_names
)

print("Global SHAP Summary (bar plot):")
shap.summary_plot(shap_explanation, plot_type='bar')

In [ ]:
print("SHAP Beeswarm Plot — direction and magnitude of feature effects:")
shap.summary_plot(shap_explanation, X_transformed, feature_names=feature_names)

In [ ]:
print("Local SHAP Explanation — first training sample:")
shap.plots.waterfall(shap_explanation[0], max_display=10)

## 12. Save Results

In [ ]:
os.makedirs("results", exist_ok=True)

summary = pd.DataFrame([
    {"Stage": "Baseline (0.5)",    "Accuracy": accuracy_score(y_test, y_pred_base_05),
     "ROC-AUC": roc_auc_score(y_test, y_prob_base),
     "Recall":  recall_score(y_test, y_pred_base_05)},
    {"Stage": "RandomCV (0.35)",   "Accuracy": accuracy_score(y_test, (y_prob_rand >= 0.35).astype(int)),
     "ROC-AUC": roc_auc_score(y_test, y_prob_rand),
     "Recall":  recall_score(y_test, (y_prob_rand >= 0.35).astype(int))},
    {"Stage": "GridSearch (0.5)",  "Accuracy": accuracy_score(y_test, best_model.predict(X_test)),
     "ROC-AUC": roc_auc_score(y_test, y_prob),
     "Recall":  recall_score(y_test, best_model.predict(X_test))},
    {"Stage": "GridSearch (0.35)", "Accuracy": accuracy_score(y_test, y_pred_adjusted),
     "ROC-AUC": roc_auc_score(y_test, y_prob),
     "Recall":  recall_score(y_test, y_pred_adjusted)},
])

summary.to_excel("results/xgb_ripas_summary.xlsx", index=False)
print("Saved: results/xgb_ripas_summary.xlsx")
display(summary)